# Reward Competition Elo rating Calculation

## Importing other Python Libraries/Modules

In [1]:
import copy
import re
import os
import sys
import string
import glob
import ast
from collections import Counter
from collections import defaultdict
import warnings
import matplotlib.pyplot as plt
import pandas as pd
os.chdir(r'c:\\Users\\megha\\Documents\\GitHub\\social_competiton_elo_rating\\')
from src.elorating import calculation_edit as calculation
from src.elorating import dataframe
# Increase size of plot in jupyter

plt.rcParams["figure.figsize"] = (18,10)

In [2]:
def get_loser(row):
    mice = row['Match'].replace(' ', '').split('.')
    mouse1, mouse2 = f'{mice[0]}.{mice[1][0]}', f'{mice[0][-1]}.{mice[-1]}'
    if str['winner'] == mouse1:
        return str(mouse2)
    else:
        return str(mouse1)
    
def get_winner(row):
    mice = row['Match'].replace(' ', '').split('.')
    mouse1, mouse2 = f'{mice[0]}.{mice[1][0]}', f'{mice[0][-1]}.{mice[-1]}'
    if row['winner'].replace(' ', '') == 'tie':
        return mouse2
    else:
        if row['winner'].replace(' ', '') != mouse1:
            if row['winner'].replace(' ', '') != mouse2:
                print(f"Warning: {row['Match']} was not scored properly, winner labeled as {row['winner']})")
        return row['winner']
    
def write_excel(output_file_path, final_elo_df, elo_dfs):
    with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
        
        # Write the final ELO scores as the first sheet
        final_elo_df.to_excel(writer, sheet_name='Final Elo Scores', index=False)
        print(f"Written sheet: 'Final Elo Scores' with {len(final_elo_df)} rows")
        
        # Write each file's ELO dataframe as separate sheets
        for file_name, elo_df in elo_dfs.items():
            # Clean the sheet name
            sheet_name = file_name.replace('.xlsx','')
            
            # Handle potential duplicate sheet names
            original_sheet_name = sheet_name
            counter = 1
            while sheet_name in [ws.title for ws in writer.book.worksheets]:
                sheet_name = f"{original_sheet_name}_{counter}"
                if len(sheet_name) > 31:
                    sheet_name = f"{original_sheet_name[:28]}_{counter}"
                counter += 1
            
            # Write the dataframe to the sheet
            elo_df.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"Written sheet: '{sheet_name}' with {len(elo_df)} rows")        

In [3]:
#input folder to all the excel sheets with urinemarking data
input_folder = r"C:\Users\megha\Documents\GitHub\social_competiton_elo_rating\jupyter_notebooks\data\nina\reward_comp"

winner_column = 'winner' #label of the winner column exactly as it apperas in your excel sheets
loser_column = 'loser' #label of the loser column exactly as it apperas in your excel sheets
tie_column = 'tie' #label of the tie column exactly as it apperas in your excel sheets if it exists
tie_flag = 'TIE' #what is written in your excel to indicate that a match is a tie in your tie column
k_factor = 40 #k factor to use for elo rating calculations 

header = 0 #row of column names in your excel sheets, 0 indexed, 0 is top
#output folder and file name to save the elo rating excel sheet
output_file_path = r"C:\Users\megha\Documents\GitHub\social_competiton_elo_rating\jupyter_notebooks\data\nina\rc_kfactor40.xlsx"

In [6]:
elo_dfs = {}
final_elo_dicts = {}
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.endswith('.xlsx'):
            file_dfs = []
            raw_data_file_path = os.path.join(input_folder, file)
            xls = pd.ExcelFile(raw_data_file_path)
            per_sheet_dataframe = pd.read_excel(raw_data_file_path, header=header)
            columns_to_drop = [col for col in per_sheet_dataframe.columns if 'tie' in col.lower()]
            df_cleaned = per_sheet_dataframe.drop(columns=columns_to_drop)

            # Step 2: Identify Trial X Winner columns
            trial_winner_cols = [col for col in df_cleaned.columns if 'trial' in col.lower() and 'winner' in col.lower()]

            # Step 3: Melt across Trial X Winner columns
            df_melted = df_cleaned.melt(
                id_vars = ['Date', 'Match'],
                value_vars=trial_winner_cols,
                var_name='trial',
                value_name='winner'
            )

            # Optional: Clean up the trial column to extract just the trial number
            df_melted['trial_number'] = df_melted['trial'].str.extract(r'Trial (\d+)')
            df_melted['winner'] = df_melted['winner'].astype(str)
            df_melted['loser'] = df_melted.apply(get_loser, axis=1)
            df_melted['loser'] = df_melted['loser'].astype(str)  
            df_melted['tie'] = df_melted['winner'].apply(lambda x: 'TIE' if x == 'tie' else '')            
            df_melted['winner'] = df_melted.apply(get_winner, axis = 1)            
            file_dfs.append(df_melted)
            final_elo_dict = defaultdict(float)
            elo_df = calculation.get_elo_df(dataframe=df_melted, winner_id_column=winner_column, loser_id_column=loser_column, tie_column=tie_column, tie_flag = tie_flag, k_factor=k_factor)
            elo_dfs[file] = elo_df
            for subject in elo_df['subject_id'].unique():
                latest_elo = calculation.get_latest_elo_for_subject(elo_df, subject)
                final_elo_dict[subject] = latest_elo
            final_elo_dicts[file] = final_elo_dict

final_elo_df = pd.DataFrame([
    {'file': file_name, 'subject': subject, 'final elo score': elo}
    for file_name, subjects_dict in final_elo_dicts.items()
    for subject, elo in subjects_dict.items()])        
write_excel(output_file_path, final_elo_df, elo_dfs)

final_elo_df

TypeError: 'type' object is not subscriptable

In [76]:
final_elo_df

,file,subject,final elo score
0,CD1_Reward_Competition_Scoring.xlsx,1.2,1181.7
1,CD1_Reward_Competition_Scoring.xlsx,1.1,993.8
2,CD1_Reward_Competition_Scoring.xlsx,1.4,717.8
3,CD1_Reward_Competition_Scoring.xlsx,1.3,1106.7
4,CD1_Reward_Competition_Scoring.xlsx,2.1,959.5
5,CD1_Reward_Competition_Scoring.xlsx,2.2,873.6
6,CD1_Reward_Competition_Scoring.xlsx,2.4,1206.8
7,CD1_Reward_Competition_Scoring.xlsx,2.3,960.1
8,Cohort_3_Spring24_CD1_Reward_Comp_Scoring (1) ...,1.2,1128.9
9,Cohort_3_Spring24_CD1_Reward_Comp_Scoring (1) ...,1.1,833.1
